In [ ]:
# -*- coding: utf-8 -*-
"""
VAE on mnist
"""

import torch
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch import nn, optim
from torch.nn import functional as F
from tqdm import tqdm
import os

# os.chdir(os.path.dirname(__file__))

'模型结构'
class Encoder(torch.nn.Module):
    #编码器，将input_size维度数据压缩为latent_size维度的mu和sigma
    def __init__(self, input_size, hidden_size, latent_size):
        super(Encoder, self).__init__()
        self.linear = torch.nn.Linear(input_size, hidden_size)
        self.mu = torch.nn.Linear(hidden_size, latent_size)
        self.sigma = torch.nn.Linear(hidden_size, latent_size)
    def forward(self, x):# x: bs,input_size
        x = F.relu(self.linear(x)) #-> bs,hidden_size
        mu = self.mu(x) #-> bs,latent_size
        sigma = self.sigma(x)#-> bs,latent_size
        return mu,sigma

class Decoder(torch.nn.Module):
    #解码器，将latent_size维度的数据转换为output_size维度的数据
    def __init__(self, latent_size, hidden_size, output_size):
        super(Decoder, self).__init__()
        self.linear1 = torch.nn.Linear(latent_size, hidden_size)
        self.linear2 = torch.nn.Linear(hidden_size, output_size)
    def forward(self, x): # x:bs,latent_size
        x = F.relu(self.linear1(x)) #->bs,hidden_size
        x = torch.sigmoid(self.linear2(x)) #->bs,output_size
        return x

class VAE(torch.nn.Module):
    #将编码器解码器组合
    def __init__(self, input_size, output_size, latent_size, hidden_size):
        super(VAE, self).__init__()
        self.encoder = Encoder(input_size, hidden_size, latent_size)
        self.decoder = Decoder(latent_size, hidden_size, output_size)
    def forward(self, x): #x: bs,input_size
        # 压缩，获取mu和sigma
        mu,sigma = self.encoder(x) #mu,sigma: bs,latent_size
        # 采样，获取采样数据
        eps = torch.randn_like(sigma)  #eps: bs,latent_size
        z = mu + eps*sigma  #z: bs,latent_size
        # 重构，根据采样数据获取重构数据
        re_x = self.decoder(z) # re_x: bs,output_size
        return re_x,mu,sigma

#损失函数
#交叉熵，衡量各个像素原始数据与重构数据的误差
loss_BCE = torch.nn.BCELoss(reduction = 'sum')
#均方误差可作为交叉熵替代使用.衡量各个像素原始数据与重构数据的误差
loss_MSE = torch.nn.MSELoss(reduction = 'sum')
#KL散度，衡量正态分布(mu,sigma)与正态分布(0,1)的差异，来源于公式计算
#这一段是直接求导过后的结果，让模型不断趋近于标准正态分布
loss_KLD = lambda mu,sigma: -0.5 * torch.sum(1 + torch.log(sigma**2) - mu.pow(2) - sigma**2)


'超参数及构造模型'
#模型参数
latent_size =16 #压缩后的特征维度
hidden_size = 128 #encoder和decoder中间层的维度
input_size= output_size = 28*28 #原始图片和生成图片的维度

#训练参数
epochs = 20 #训练时期
batch_size = 32 #每步训练样本数
learning_rate = 1e-4 #学习率
device =torch.device('cuda' if torch.cuda.is_available() else 'cpu')#训练设备

#确定模型，导入已训练模型（如有）
modelname = 'vae.pth'
model = VAE(input_size,output_size,latent_size,hidden_size).to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
try:
    model.load_state_dict(torch.load(modelname))
    print('[INFO] Load Model complete')
except:
    pass



'训练模型'
#准备mnist数据集 (数据会下载到py文件所在的data文件夹下)
train_loader = torch.utils.data.DataLoader(
    datasets.MNIST('./data', train=True, download=True,
                   transform=transforms.ToTensor()),
    batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(
    datasets.MNIST('./data', train=False, transform=transforms.ToTensor()),
    batch_size=batch_size, shuffle=False)

#训练及测试
loss_history = {'train':[],'eval':[]}
for epoch in range(epochs):   
    #训练
    model.train()
    #每个epoch重置损失，设置进度条
    train_loss = 0
    train_nsample = 0
    t = tqdm(train_loader,desc = f'[train]epoch:{epoch}')
    for imgs, lbls in t: #imgs:(bs,28,28)
        bs = imgs.shape[0]
        #获取数据
        imgs = imgs.to(device).view(bs,input_size) #imgs:(bs,28*28)
        #模型运算     
        re_imgs, mu, sigma = model(imgs)
        #计算损失
        loss_re = loss_BCE(re_imgs, imgs) # 重构与原始数据的差距(也可使用loss_MSE)
        loss_norm = loss_KLD(mu, sigma) # 正态分布(mu,sigma)与正态分布(0,1)的差距
        loss = loss_re + loss_norm
        #反向传播、参数优化，重置
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        #计算平均损失，设置进度条
        train_loss += loss.item()
        train_nsample += bs
        t.set_postfix({'loss':train_loss/train_nsample})
    #每个epoch记录总损失
    loss_history['train'].append(train_loss/train_nsample)

    #测试
    model.eval()
    #每个epoch重置损失，设置进度条
    test_loss = 0
    test_nsample = 0
    e = tqdm(test_loader,desc = f'[eval]epoch:{epoch}')
    for imgs, label in e:
        bs = imgs.shape[0]
        #获取数据
        imgs = imgs.to(device).view(bs,input_size)
        #模型运算   
        re_imgs, mu, sigma = model(imgs)
        #计算损失
        loss_re = loss_BCE(re_imgs, imgs) 
        loss_norm = loss_KLD(mu, sigma) 
        loss = loss_re + loss_norm
        #计算平均损失，设置进度条
        test_loss += loss.item()
        test_nsample += bs
        e.set_postfix({'loss':test_loss/test_nsample})
    #每个epoch记录总损失    
    loss_history['eval'].append(test_loss/test_nsample)


    #展示效果   
    #按标准正态分布取样来自造数据
    sample = torch.randn(1,latent_size).to(device)
    #用decoder生成新数据
    gen = model.decoder(sample)[0].view(28,28)
    #将测试步骤中的真实数据、重构数据和上述生成的新数据绘图
    concat = torch.cat((imgs[0].view(28, 28),
            re_imgs[0].view( 28, 28), gen), 1)
    plt.matshow(concat.cpu().detach().numpy())
    plt.show()

    #显示每个epoch的loss变化
    plt.plot(range(epoch+1),loss_history['train'])
    plt.plot(range(epoch+1),loss_history['eval'])
    plt.show()
    #存储模型
    torch.save(model.state_dict(),modelname)

'调用模型'
#按标准正态分布取样来自造数据
sample = torch.randn(1,latent_size).to(device)
#用decoder生成新数据
generate = model.decoder(sample)[0].view(28,28)
#展示生成数据
plt.matshow(generate.cpu().detach().numpy())
plt.show()

In [ ]:
# -*- coding: utf-8 -*-
"""
VAE on MNIST (仅使用标签为9的数据，生成数字9)
"""

import torch
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch import nn, optim
from torch.nn import functional as F
from tqdm import tqdm
import os

# os.chdir(os.path.dirname(__file__))  # 若需指定代码所在目录，可取消注释

'模型结构（无修改，保持原VAE结构）'
class Encoder(torch.nn.Module):
    def __init__(self, input_size, hidden_size, latent_size):
        super(Encoder, self).__init__()
        self.linear = torch.nn.Linear(input_size, hidden_size)
        self.mu = torch.nn.Linear(hidden_size, latent_size)
        self.sigma = torch.nn.Linear(hidden_size, latent_size)
    
    def forward(self, x):  # x: bs, input_size
        x = F.relu(self.linear(x))  # -> bs, hidden_size
        mu = self.mu(x)  # -> bs, latent_size
        sigma = self.sigma(x)  # -> bs, latent_size
        return mu, sigma

class Decoder(torch.nn.Module):
    def __init__(self, latent_size, hidden_size, output_size):
        super(Decoder, self).__init__()
        self.linear1 = torch.nn.Linear(latent_size, hidden_size)
        self.linear2 = torch.nn.Linear(hidden_size, output_size)
    
    def forward(self, x):  # x: bs, latent_size
        x = F.relu(self.linear1(x))  # -> bs, hidden_size
        x = torch.sigmoid(self.linear2(x))  # -> bs, output_size（与输入像素范围[0,1]匹配）
        return x

class VAE(torch.nn.Module):
    def __init__(self, input_size, output_size, latent_size, hidden_size):
        super(VAE, self).__init__()
        self.encoder = Encoder(input_size, hidden_size, latent_size)
        self.decoder = Decoder(latent_size, hidden_size, output_size)
    
    def forward(self, x):  # x: bs, input_size
        mu, sigma = self.encoder(x)  # 编码得到分布参数
        eps = torch.randn_like(sigma)  # 重参数化：生成标准正态噪声
        z = mu + eps * sigma  # 采样潜在向量
        re_x = self.decoder(z)  # 解码重构输入
        return re_x, mu, sigma

'损失函数（无修改，保持原定义）'
loss_BCE = torch.nn.BCELoss(reduction='sum')  # 重构损失：匹配像素级差异
loss_KLD = lambda mu, sigma: -0.5 * torch.sum(1 + torch.log(sigma**2) - mu.pow(2) - sigma**2)  # KL散度：约束潜在分布

'超参数及模型初始化（无修改）'
# 模型参数
latent_size = 16    # 潜在空间维度
hidden_size = 128   # 编码器/解码器隐藏层维度
input_size = output_size = 28 * 28  # MNIST图像扁平化后维度（28x28）

# 训练参数
epochs = 50         # 训练轮次
batch_size = 32     # 批次大小
learning_rate = 1e-4  # 学习率
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # 自动选择GPU/CPU

# 模型加载与优化器
modelname = 'vae_mnist_9.pth'  # 模型保存名（区分原模型，避免覆盖）
model = VAE(input_size, output_size, latent_size, hidden_size).to(device)
optimizer = Adam(model.parameters(), lr=learning_rate)

# 加载已训练模型（若有）
try:
    model.load_state_dict(torch.load(modelname))
    print(f'[INFO] 成功加载模型：{modelname}')
except:
    print('[INFO] 未找到已训练模型，将从头训练')


'核心修改：筛选MNIST中标签为9的样本'
def get_mnist_label_9(data_root, train=True, transform=None):
    """
    加载MNIST数据集，并筛选出标签为9的样本
    :param data_root: 数据集保存路径
    :param train: True=训练集，False=测试集
    :param transform: 数据预处理方式
    :return: 仅含标签9的数据集
    """
    # 加载完整MNIST数据集
    full_dataset = datasets.MNIST(
        root=data_root,
        train=train,
        download=True,
        transform=transform
    )
    
    # 筛选标签为9的样本索引（full_dataset.targets是所有样本的标签）
    label_9_indices = torch.where(full_dataset.targets == 9)[0]  # 找到标签=9的位置
    
    # 根据索引提取子数据集（仅含标签9的样本）
    subset = torch.utils.data.Subset(full_dataset, label_9_indices)
    return subset

# 数据预处理（与原代码一致：转为Tensor并归一化到[0,1]）
transform = transforms.ToTensor()

# 加载“仅含标签9”的训练集和测试集（关键修改点）
train_dataset = get_mnist_label_9(
    data_root='./data',  # 数据集保存路径（与原代码一致）
    train=True,
    transform=transform
)
test_dataset = get_mnist_label_9(
    data_root='./data',
    train=False,
    transform=transform
)

# 创建数据加载器（与原代码一致，批量加载数据）
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True  # 训练集打乱，提升泛化能力
)
test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False  # 测试集无需打乱
)

# 打印数据集信息（验证是否仅含标签9的样本）
print(f'[INFO] 训练集：共{len(train_dataset)}个样本（均为标签9）')
print(f'[INFO] 测试集：共{len(test_dataset)}个样本（均为标签9）')


'训练与测试流程（仅适配筛选后的数据，逻辑无修改）'
loss_history = {'train': [], 'eval': []}  # 记录损失变化

for epoch in range(epochs):
    # ---------------------- 训练阶段 ----------------------
    model.train()  # 开启训练模式（启用Dropout等，此处无Dropout，仅规范流程）
    train_loss = 0.0
    train_nsample = 0
    
    # 进度条显示训练过程
    t_bar = tqdm(train_loader, desc=f'[训练] 第{epoch+1}/{epochs}轮')
    for imgs, lbls in t_bar:
        # 数据预处理：展平图像（28x28 -> 784）并转移到设备
        bs = imgs.shape[0]  # 当前批次大小
        imgs = imgs.to(device).view(bs, input_size)  # imgs: [bs, 784]
        
        # 模型前向传播
        re_imgs, mu, sigma = model(imgs)
        
        # 计算损失（重构损失 + KL散度损失）
        loss_re = loss_BCE(re_imgs, imgs)  # 重构损失：衡量像素差异
        loss_norm = loss_KLD(mu, sigma)   # KL损失：约束潜在分布接近标准正态
        total_loss = loss_re + loss_norm
        
        # 反向传播与参数更新
        optimizer.zero_grad()  # 清空梯度（避免累积）
        total_loss.backward()  # 计算梯度
        optimizer.step()       # 更新模型参数
        
        # 累计损失与样本数，更新进度条
        train_loss += total_loss.item()
        train_nsample += bs
        t_bar.set_postfix({'平均损失': train_loss / train_nsample})
    
    # 记录当前轮训练损失
    loss_history['train'].append(train_loss / train_nsample)

    # ---------------------- 测试阶段 ----------------------
    model.eval()  # 开启评估模式（禁用Dropout等）
    test_loss = 0.0
    test_nsample = 0
    
    # 进度条显示测试过程
    e_bar = tqdm(test_loader, desc=f'[测试] 第{epoch+1}/{epochs}轮')
    with torch.no_grad():  # 测试阶段不计算梯度，节省内存
        for imgs, lbls in e_bar:
            bs = imgs.shape[0]
            imgs = imgs.to(device).view(bs, input_size)
            
            # 模型前向传播
            re_imgs, mu, sigma = model(imgs)
            
            # 计算损失（与训练阶段一致）
            loss_re = loss_BCE(re_imgs, imgs)
            loss_norm = loss_KLD(mu, sigma)
            total_loss = loss_re + loss_norm
            
            # 累计测试损失
            test_loss += total_loss.item()
            test_nsample += bs
            e_bar.set_postfix({'平均损失': test_loss / test_nsample})
    
    # 记录当前轮测试损失
    loss_history['eval'].append(test_loss / test_nsample)

    # ---------------------- 结果展示 ----------------------
    # 1. 可视化：真实图像（标签9）、重构图像、新生成图像
    with torch.no_grad():
        # 随机生成潜在向量（标准正态分布，与训练时KL约束的目标分布一致）
        sample_z = torch.randn(1, latent_size).to(device)
        gen_img = model.decoder(sample_z)[0].view(28, 28)  # 生成图像（展平->28x28）
        
        # 取测试集中第一个样本作为“真实图像”和“重构图像”示例
        real_img = imgs[0].view(28, 28)  # 真实图像（标签9）
        recon_img = re_imgs[0].view(28, 28)  # 重构图像（基于真实图像编码解码）
        
        # 拼接三张图（横向排列：真实→重构→生成）
        concat_img = torch.cat([real_img, recon_img, gen_img], dim=1)
        plt.matshow(concat_img.cpu().numpy(), cmap='gray')  # 转为numpy并显示（灰度图）
        plt.title('真实图像（9） | 重构图像 | 新生成图像（9）')
        plt.axis('off')  # 隐藏坐标轴
        plt.show()

    # 2. 可视化：训练/测试损失变化
    plt.plot(range(epoch+1), loss_history['train'], label='训练损失')
    plt.plot(range(epoch+1), loss_history['eval'], label='测试损失')
    plt.xlabel('训练轮次（Epoch）')
    plt.ylabel('平均损失')
    plt.legend()
    plt.title('VAE训练损失变化（仅用标签9数据）')
    plt.show()

    # ---------------------- 模型保存 ----------------------
    torch.save(model.state_dict(), modelname)
    print(f'[INFO] 模型已保存为：{modelname}\n')


'单独调用模型生成数字9（无修改）'
print('[INFO] 单独生成数字9...')
with torch.no_grad():
    # 随机采样潜在向量（标准正态分布）
    gen_z = torch.randn(1, latent_size).to(device)
    final_gen_img = model.decoder(gen_z)[0].view(28, 28)
    
    # 显示生成结果
    plt.matshow(final_gen_img.cpu().numpy(), cmap='gray')
    plt.title('单独生成的数字9')
    plt.axis('off')
    plt.show()

In [ ]:
# -*- coding: utf-8 -*-
"""
CVAE on mnist
条件变分自编码器在MNIST上的实现
"""

import torch
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch import nn, optim
from torch.nn import functional as F
from tqdm import tqdm
import os

# os.chdir(os.path.dirname(__file__))

'模型结构'
class Encoder(torch.nn.Module):
    # 编码器，将input_size维度数据和条件信息压缩为latent_size维度的mu和sigma
    def __init__(self, input_size, condition_size, hidden_size, latent_size):
        super(Encoder, self).__init__()
        # 输入是原始数据+条件信息的拼接
        self.linear1 = torch.nn.Linear(input_size + condition_size, hidden_size)
        self.linear2 = torch.nn.Linear(hidden_size, hidden_size)
        self.mu = torch.nn.Linear(hidden_size, latent_size)
        self.sigma = torch.nn.Linear(hidden_size, latent_size)
        
    def forward(self, x, c):  # x: bs,input_size; c: bs,condition_size
        # 将输入数据和条件信息拼接
        x = torch.cat([x, c], dim=1)  # -> bs, input_size + condition_size
        x = F.relu(self.linear1(x))   # -> bs, hidden_size
        x = F.relu(self.linear2(x))   # -> bs, hidden_size
        mu = self.mu(x)               # -> bs, latent_size
        sigma = self.sigma(x)         # -> bs, latent_size
        # 确保sigma为正，使用softplus激活函数
        sigma = F.softplus(sigma) + 1e-6  # 防止sigma为0
        return mu, sigma

class Decoder(torch.nn.Module):
    # 解码器，将latent_size维度的数据和条件信息转换为output_size维度的数据
    def __init__(self, latent_size, condition_size, hidden_size, output_size):
        super(Decoder, self).__init__()
        # 输入是隐变量+条件信息的拼接
        self.linear1 = torch.nn.Linear(latent_size + condition_size, hidden_size)
        self.linear2 = torch.nn.Linear(hidden_size, hidden_size)
        self.linear3 = torch.nn.Linear(hidden_size, output_size)
        
    def forward(self, z, c):  # z: bs,latent_size; c: bs,condition_size
        # 将隐变量和条件信息拼接
        x = torch.cat([z, c], dim=1)  # -> bs, latent_size + condition_size
        x = F.relu(self.linear1(x))   # -> bs, hidden_size
        x = F.relu(self.linear2(x))   # -> bs, hidden_size
        x = torch.sigmoid(self.linear3(x))  # -> bs, output_size，确保输出在0-1之间
        return x

class CVAE(torch.nn.Module):
    # 将编码器解码器组合
    def __init__(self, input_size, output_size, condition_size, latent_size, hidden_size):
        super(CVAE, self).__init__()
        self.encoder = Encoder(input_size, condition_size, hidden_size, latent_size)
        self.decoder = Decoder(latent_size, condition_size, hidden_size, output_size)
        
    def forward(self, x, c):  # x: bs,input_size; c: bs,condition_size
        # 压缩，获取mu和sigma
        mu, sigma = self.encoder(x, c)  # mu,sigma: bs,latent_size
        # 采样，获取采样数据（重参数化技巧）
        eps = torch.randn_like(sigma)   # eps: bs,latent_size
        z = mu + eps * sigma            # z: bs,latent_size
        # 重构，根据采样数据和条件信息获取重构数据
        re_x = self.decoder(z, c)       # re_x: bs,output_size
        return re_x, mu, sigma
    
    def generate(self, c, z=None):
        # 生成指定条件的数据
        if z is None:
            # 如果没有提供z，则从标准正态分布采样
            z = torch.randn(c.size(0), self.encoder.mu.out_features).to(c.device)
        return self.decoder(z, c)

# 损失函数
# 交叉熵，衡量各个像素原始数据与重构数据的误差
loss_BCE = torch.nn.BCELoss(reduction='sum')
# 均方误差可作为交叉熵替代使用.衡量各个像素原始数据与重构数据的误差
loss_MSE = torch.nn.MSELoss(reduction='sum')
# KL散度，衡量正态分布(mu,sigma)与正态分布(0,1)的差异
loss_KLD = lambda mu, sigma: -0.5 * torch.sum(1 + torch.log(sigma**2) - mu.pow(2) - sigma**2)


'超参数及构造模型'
# 模型参数
latent_size = 16          # 压缩后的特征维度
hidden_size = 128         # encoder和decoder中间层的维度
input_size = output_size = 28*28  # 原始图片和生成图片的维度
condition_size = 10       # 条件信息维度（MNIST有10个类别）

# 训练参数
epochs = 20               # 训练时期
batch_size = 32           # 每步训练样本数
learning_rate = 1e-4      # 学习率
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # 训练设备

# 确定模型，导入已训练模型（如有）
modelname = 'cvae.pth'
model = CVAE(input_size, output_size, condition_size, latent_size, hidden_size).to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
try:
    model.load_state_dict(torch.load(modelname))
    print('[INFO] Load Model complete')
except:
    print('[INFO] No existing model, starting from scratch')


'训练模型'
# 准备mnist数据集 (数据会下载到py文件所在的data文件夹下)
train_loader = torch.utils.data.DataLoader(
    datasets.MNIST('./data', train=True, download=True,
                   transform=transforms.ToTensor()),
    batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(
    datasets.MNIST('./data', train=False, transform=transforms.ToTensor()),
    batch_size=batch_size, shuffle=False)

# 工具函数：将标签转换为独热编码
def to_one_hot(labels, num_classes=10):
    return torch.eye(num_classes, device=labels.device)[labels]

# 训练及测试
loss_history = {'train': [], 'eval': []}
for epoch in range(epochs):   
    # 训练
    model.train()
    # 每个epoch重置损失，设置进度条
    train_loss = 0
    train_nsample = 0
    t = tqdm(train_loader, desc=f'[train]epoch:{epoch}')
    for imgs, lbls in t:  # imgs:(bs,1,28,28), lbls:(bs,)
        bs = imgs.shape[0]
        # 获取数据并预处理
        imgs = imgs.to(device).view(bs, input_size)  # imgs:(bs,28*28)
        lbls = lbls.to(device)
        # 将标签转换为独热编码作为条件信息
        conditions = to_one_hot(lbls, num_classes=condition_size)  # (bs, 10)
        
        # 模型运算     
        re_imgs, mu, sigma = model(imgs, conditions)
        
        # 计算损失
        loss_re = loss_BCE(re_imgs, imgs)  # 重构与原始数据的差距
        loss_norm = loss_KLD(mu, sigma)    # 正态分布(mu,sigma)与标准正态分布的差距
        loss = loss_re + loss_norm
        
        # 反向传播、参数优化，重置
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        # 计算平均损失，设置进度条
        train_loss += loss.item()
        train_nsample += bs
        t.set_postfix({'loss': train_loss / train_nsample})
    
    # 每个epoch记录总损失
    loss_history['train'].append(train_loss / train_nsample)

    # 测试
    model.eval()
    # 每个epoch重置损失，设置进度条
    test_loss = 0
    test_nsample = 0
    e = tqdm(test_loader, desc=f'[eval]epoch:{epoch}')
    with torch.no_grad():  # 测试时不计算梯度
        for imgs, lbls in e:
            bs = imgs.shape[0]
            # 获取数据并预处理
            imgs = imgs.to(device).view(bs, input_size)
            lbls = lbls.to(device)
            conditions = to_one_hot(lbls, num_classes=condition_size)
            
            # 模型运算   
            re_imgs, mu, sigma = model(imgs, conditions)
            
            # 计算损失
            loss_re = loss_BCE(re_imgs, imgs) 
            loss_norm = loss_KLD(mu, sigma) 
            loss = loss_re + loss_norm
            
            # 计算平均损失，设置进度条
            test_loss += loss.item()
            test_nsample += bs
            e.set_postfix({'loss': test_loss / test_nsample})
    
    # 每个epoch记录总损失    
    loss_history['eval'].append(test_loss / test_nsample)

    # 展示效果   
    # 生成0-9的条件
    sample_conditions = to_one_hot(torch.arange(10, device=device), num_classes=condition_size)
    # 生成新数据
    generated = model.generate(sample_conditions)
    
    # 绘制生成的数字
    plt.figure(figsize=(10, 1))
    for i in range(10):
        plt.subplot(1, 10, i+1)
        plt.imshow(generated[i].cpu().detach().view(28, 28).numpy(), cmap='gray')
        plt.axis('off')
        plt.title(str(i))
    plt.suptitle(f'Generated digits (epoch {epoch})')
    plt.show()

    # 显示每个epoch的loss变化
    plt.figure(figsize=(10, 4))
    plt.plot(range(epoch+1), loss_history['train'], label='Train Loss')
    plt.plot(range(epoch+1), loss_history['eval'], label='Eval Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Evaluation Loss')
    plt.legend()
    plt.show()
    
    # 存储模型
    torch.save(model.state_dict(), modelname)

'调用模型生成特定数字'
# 生成指定数字（0-9）
model.eval()
with torch.no_grad():
    # 创建0-9的条件
    conditions = to_one_hot(torch.arange(10, device=device), num_classes=condition_size)
    # 生成数据
    generated = model.generate(conditions)
    
    # 显示生成结果
    plt.figure(figsize=(10, 1))
    for i in range(10):
        plt.subplot(1, 10, i+1)
        plt.imshow(generated[i].cpu().detach().view(28, 28).numpy(), cmap='gray')
        plt.axis('off')
        plt.title(f'Digit {i}')
    plt.tight_layout()
    plt.show()

# 生成同一数字的不同变体
digit_to_generate = 5  # 选择要生成的数字
num_variations = 5     # 生成的变体数量

model.eval()
with torch.no_grad():
    # 创建重复的条件
    conditions = to_one_hot(torch.tensor([digit_to_generate]*num_variations, device=device), 
                           num_classes=condition_size)
    # 生成数据
    generated = model.generate(conditions)
    
    # 显示生成结果
    plt.figure(figsize=(10, 2))
    for i in range(num_variations):
        plt.subplot(1, num_variations, i+1)
        plt.imshow(generated[i].cpu().detach().view(28, 28).numpy(), cmap='gray')
        plt.axis('off')
    plt.suptitle(f'Variations of digit {digit_to_generate}')
    plt.tight_layout(rect=[0, 0, 1, 0.95])  # 为标题留出空间
    plt.show()
